# Retos · Proyecto gestor de tareas

Construir paso a paso un pequeño programa de terminal, `tareas.py`, que guarda una lista de tareas con prioridad y fecha límite. Reúne lo visto en otros temas: `dataclasses` y `enum` para el modelo, `datetime` para las fechas, JSON para la persistencia, `argparse` con subcomandos (`añadir`, `listar`, `completar` y `borrar`), `logging` para los mensajes y `pytest` para los tests. El módulo se escribe por partes con `%%writefile` (`-a` añade al final del fichero), se prueba importándolo y, al final, se ejecuta como programa con `subprocess`.

In [1]:
import importlib
import json
import os
import shutil
import subprocess
import sys
from datetime import date

sys.dont_write_bytecode = True  # sin __pycache__ junto al notebook

def recarga():
    # vuelve a importar tareas.py para ver lo último que se ha escrito
    sys.modules.pop("tareas", None)
    return importlib.import_module("tareas")

HOY = date(2026, 11, 6)  # fecha fija para que las salidas no dependan del día en que se ejecute

## 1. Modelo: `Prioridad` y `Tarea`

Empezar `tareas.py` con los imports, un *logger* propio, la enumeración `Prioridad` (`IntEnum`, para poder compararlas y ordenarlas) y la *dataclass* `Tarea` con `id`, `titulo`, `prioridad`, `limite` (una `date` o `None`) y `hecha`. Añadir `a_dict` y `de_dict` para convertirla a tipos que entiende JSON (la prioridad por su nombre y la fecha en ISO 8601).

In [2]:
%%writefile tareas.py
import argparse
import json
import logging
import os
import sys
from dataclasses import asdict, dataclass
from datetime import date, timedelta
from enum import IntEnum
from pathlib import Path

log = logging.getLogger("tareas")


class Prioridad(IntEnum):
    BAJA = 1
    MEDIA = 2
    ALTA = 3


@dataclass
class Tarea:
    id: int
    titulo: str
    prioridad: Prioridad = Prioridad.MEDIA
    limite: date | None = None
    hecha: bool = False

    def a_dict(self):
        return asdict(self) | {
            "prioridad": self.prioridad.name,
            "limite": self.limite.isoformat() if self.limite else None,
        }

    @classmethod
    def de_dict(cls, datos):
        limite = date.fromisoformat(datos["limite"]) if datos["limite"] else None
        return cls(datos["id"], datos["titulo"], Prioridad[datos["prioridad"]], limite, datos["hecha"])

Writing tareas.py


In [3]:
tareas = recarga()
tarea = tareas.Tarea(1, "Repasar grafos", tareas.Prioridad.ALTA, date(2026, 11, 8))
print(tarea)
print(tarea.a_dict())
print(tareas.Prioridad.ALTA > tareas.Prioridad.BAJA, tareas.Prioridad["MEDIA"], [p.name for p in tareas.Prioridad])
assert tareas.Tarea.de_dict(tarea.a_dict()) == tarea

Tarea(id=1, titulo='Repasar grafos', prioridad=<Prioridad.ALTA: 3>, limite=datetime.date(2026, 11, 8), hecha=False)
{'id': 1, 'titulo': 'Repasar grafos', 'prioridad': 'ALTA', 'limite': '2026-11-08', 'hecha': False}
True 2 ['BAJA', 'MEDIA', 'ALTA']


## 2. Fechas límite con `datetime`

Añadir `lee_fecha(texto, hoy)`, que acepta `AAAA-MM-DD`, `hoy`, `mañana` o `+N` (dentro de $N$ días) y lanza `ValueError` con cualquier otra cosa, y `describe_limite(limite, hoy)`, que devuelve textos como «vence en 3 días», «vence hoy» o «vencida hace 1 día». La fecha de hoy se pasa como argumento para poder probarlas con una fecha fija.

In [4]:
%%writefile -a tareas.py


def lee_fecha(texto, hoy):
    texto = texto.strip().lower()
    if texto == "hoy":
        return hoy
    if texto == "mañana":
        return hoy + timedelta(days=1)
    if texto.startswith("+") and texto[1:].isdigit():
        return hoy + timedelta(days=int(texto[1:]))
    return date.fromisoformat(texto)


def describe_limite(limite, hoy):
    if limite is None:
        return "sin fecha"
    dias = (limite - hoy).days
    plural = "" if abs(dias) == 1 else "s"
    if dias == 0:
        return "vence hoy"
    if dias > 0:
        return f"vence en {dias} día{plural}"
    return f"vencida hace {-dias} día{plural}"

Appending to tareas.py


In [5]:
tareas = recarga()
for texto in ["hoy", "mañana", "+30", "2027-01-06"]:
    limite = tareas.lee_fecha(texto, HOY)
    print(f"{texto:>10} -> {limite}  ({tareas.describe_limite(limite, HOY)})")
print(tareas.describe_limite(date(2026, 11, 5), HOY), "|", tareas.describe_limite(None, HOY))

try:
    tareas.lee_fecha("el lunes", HOY)
except ValueError as error:
    print("ValueError:", error)

       hoy -> 2026-11-06  (vence hoy)
    mañana -> 2026-11-07  (vence en 1 día)
       +30 -> 2026-12-06  (vence en 30 días)
2027-01-06 -> 2027-01-06  (vence en 61 días)
vencida hace 1 día | sin fecha
ValueError: Invalid isoformat string: 'el lunes'


## 3. Persistencia en JSON

Añadir `carga(ruta)`, que devuelve la lista de tareas guardada (vacía si el fichero no existe), y `guarda(tareas, ruta)`, que la escribe en JSON legible (`indent`, `ensure_ascii=False`). Para no dejar un fichero a medias si el programa se interrumpe, escribir primero en un temporal y sustituir el bueno con `os.replace`, que es atómico.

In [6]:
%%writefile -a tareas.py


def carga(ruta):
    ruta = Path(ruta)
    if not ruta.exists():
        return []
    with ruta.open(encoding="utf-8") as f:
        return [Tarea.de_dict(datos) for datos in json.load(f)]


def guarda(tareas, ruta):
    ruta = Path(ruta)
    temporal = ruta.with_suffix(".tmp")
    texto = json.dumps([t.a_dict() for t in tareas], ensure_ascii=False, indent=2)
    temporal.write_text(texto + "\n", encoding="utf-8")
    os.replace(temporal, ruta)
    log.debug("guardadas %d tareas en %s", len(tareas), ruta)

Appending to tareas.py


In [7]:
tareas = recarga()
lista = [tareas.Tarea(1, "Repasar grafos", tareas.Prioridad.ALTA, date(2026, 11, 8)),
         tareas.Tarea(2, "Comprar café", hecha=True)]
tareas.guarda(lista, "prueba.json")
with open("prueba.json", encoding="utf-8") as f:
    print(f.read())
assert tareas.carga("prueba.json") == lista
assert tareas.carga("no-existe.json") == []
os.remove("prueba.json")

[
  {
    "id": 1,
    "titulo": "Repasar grafos",
    "prioridad": "ALTA",
    "limite": "2026-11-08",
    "hecha": false
  },
  {
    "id": 2,
    "titulo": "Comprar café",
    "prioridad": "MEDIA",
    "limite": null,
    "hecha": true
  }
]



## 4. Operaciones sobre la lista

Añadir `agrega` (el nuevo `id` es el mayor más uno), `completa` y `borra`, que lanzan la excepción propia `TareaNoEncontrada` si el `id` no existe, y `ordena`, que pone primero las pendientes y después ordena por fecha límite (las que no tienen van al final) y por prioridad descendente. Cada cambio deja un mensaje `INFO` en el *logger*.

In [8]:
%%writefile -a tareas.py


class TareaNoEncontrada(LookupError):
    pass


def agrega(tareas, titulo, prioridad=Prioridad.MEDIA, limite=None):
    tarea = Tarea(max((t.id for t in tareas), default=0) + 1, titulo, prioridad, limite)
    tareas.append(tarea)
    log.info("añadida la tarea %d: %s", tarea.id, titulo)
    return tarea


def busca(tareas, id_):
    for tarea in tareas:
        if tarea.id == id_:
            return tarea
    raise TareaNoEncontrada(f"no existe la tarea {id_}")


def completa(tareas, id_):
    tarea = busca(tareas, id_)
    tarea.hecha = True
    log.info("completada la tarea %d", id_)
    return tarea


def borra(tareas, id_):
    tarea = busca(tareas, id_)
    tareas.remove(tarea)
    log.info("borrada la tarea %d", id_)
    return tarea


def ordena(tareas):
    return sorted(tareas, key=lambda t: (t.hecha, t.limite or date.max, -t.prioridad, t.id))

Appending to tareas.py


In [9]:
tareas = recarga()
lista = []
tareas.agrega(lista, "Leer sobre dataclasses")
tareas.agrega(lista, "Repasar grafos", tareas.Prioridad.ALTA, date(2026, 11, 8))
tareas.agrega(lista, "Entregar la práctica", tareas.Prioridad.BAJA, date(2026, 11, 8))
tareas.completa(lista, 1)
tareas.borra(lista, 3)
tareas.agrega(lista, "Comprar café", tareas.Prioridad.BAJA, HOY)
for t in tareas.ordena(lista):
    print(t)
assert [t.id for t in lista] == [1, 2, 3]

try:
    tareas.completa(lista, 9)
except tareas.TareaNoEncontrada as error:
    print("TareaNoEncontrada:", error)

Tarea(id=3, titulo='Comprar café', prioridad=<Prioridad.BAJA: 1>, limite=datetime.date(2026, 11, 6), hecha=False)
Tarea(id=2, titulo='Repasar grafos', prioridad=<Prioridad.ALTA: 3>, limite=datetime.date(2026, 11, 8), hecha=False)
Tarea(id=1, titulo='Leer sobre dataclasses', prioridad=<Prioridad.MEDIA: 2>, limite=None, hecha=True)
TareaNoEncontrada: no existe la tarea 9


## 5. Formato del listado

Añadir `formatea(tarea, hoy)`, que convierte una tarea en una línea de texto con una casilla `[ ]` o `[x]`, el `id`, la prioridad y el título y, si está pendiente, cuándo vence. Usar especificadores de formato para que las columnas queden alineadas.

In [10]:
%%writefile -a tareas.py


def formatea(tarea, hoy):
    marca = "x" if tarea.hecha else " "
    linea = f"[{marca}] {tarea.id:>2}  {tarea.prioridad.name:<5}  {tarea.titulo}"
    return linea if tarea.hecha else f"{linea}  ({describe_limite(tarea.limite, hoy)})"

Appending to tareas.py


In [11]:
tareas = recarga()
lista = [tareas.Tarea(1, "Leer sobre dataclasses", hecha=True),
         tareas.Tarea(2, "Repasar grafos", tareas.Prioridad.ALTA, date(2026, 11, 8)),
         tareas.Tarea(3, "Comprar café", tareas.Prioridad.BAJA, HOY),
         tareas.Tarea(12, "Entregar la práctica", tareas.Prioridad.ALTA, date(2026, 11, 1))]
for t in tareas.ordena(lista):
    print(tareas.formatea(t, HOY))

[ ] 12  ALTA   Entregar la práctica  (vencida hace 5 días)
[ ]  3  BAJA   Comprar café  (vence hoy)
[ ]  2  ALTA   Repasar grafos  (vence en 2 días)
[x]  1  MEDIA  Leer sobre dataclasses


## 6. Línea de órdenes con `argparse`

Añadir `crea_parser()` con las opciones globales `--fichero`, `--hoy` (fecha de referencia, para las pruebas) y `-v` (que se puede repetir), y un subcomando por orden con `add_subparsers`: `añadir` con el título y las opciones `-p/--prioridad` y `-l/--limite`, `listar` con `--pendientes` y `-p`, y `completar` y `borrar` con el `id`. Probarlo con `parse_args` y ver la ayuda que genera y el error de una prioridad que no existe.

In [12]:
%%writefile -a tareas.py


PRIORIDADES = [p.name for p in Prioridad]


def crea_parser():
    parser = argparse.ArgumentParser(prog="tareas", description="Gestor de tareas de terminal.")
    parser.add_argument("--fichero", default="tareas.json", help="fichero JSON con las tareas")
    parser.add_argument("--hoy", type=date.fromisoformat, default=date.today(), help="fecha de referencia")
    parser.add_argument("-v", "--verbose", action="count", default=0, help="más mensajes (-v, -vv)")
    ordenes = parser.add_subparsers(dest="orden", required=True)

    orden = ordenes.add_parser("añadir", help="añade una tarea")
    orden.add_argument("titulo")
    orden.add_argument("-p", "--prioridad", type=str.upper, choices=PRIORIDADES, default="MEDIA")
    orden.add_argument("-l", "--limite", help="AAAA-MM-DD, hoy, mañana o +N días")

    orden = ordenes.add_parser("listar", help="muestra las tareas")
    orden.add_argument("--pendientes", action="store_true", help="oculta las completadas")
    orden.add_argument("-p", "--prioridad", type=str.upper, choices=PRIORIDADES, help="solo esa prioridad")

    for nombre, ayuda in [("completar", "marca una tarea como hecha"), ("borrar", "borra una tarea")]:
        orden = ordenes.add_parser(nombre, help=ayuda)
        orden.add_argument("id", type=int)
    return parser

Appending to tareas.py


In [13]:
tareas = recarga()
parser = tareas.crea_parser()
print(parser.parse_args(["añadir", "Repasar grafos", "-p", "alta", "-l", "+2"]))
print(parser.parse_args(["--hoy", "2026-11-06", "-vv", "listar", "--pendientes"]))
print(parser.format_help())

try:
    parser.parse_args(["añadir", "Algo", "-p", "urgente"])
except SystemExit as error:
    print("SystemExit:", error.code)

Namespace(fichero='tareas.json', hoy=datetime.date(2026, 11, 6), verbose=0, orden='añadir', titulo='Repasar grafos', prioridad='ALTA', limite='+2')
Namespace(fichero='tareas.json', hoy=datetime.date(2026, 11, 6), verbose=2, orden='listar', pendientes=True, prioridad=None)
usage: tareas [-h] [--fichero FICHERO] [--hoy HOY] [-v]
              {añadir,listar,completar,borrar} ...

Gestor de tareas de terminal.

positional arguments:
  {añadir,listar,completar,borrar}
    añadir              añade una tarea
    listar              muestra las tareas
    completar           marca una tarea como hecha
    borrar              borra una tarea

options:
  -h, --help            show this help message and exit
  --fichero FICHERO     fichero JSON con las tareas
  --hoy HOY             fecha de referencia
  -v, --verbose         más mensajes (-v, -vv)

SystemExit: 2


usage: tareas añadir [-h] [-p {BAJA,MEDIA,ALTA}] [-l LIMITE] titulo
tareas añadir: error: argument -p/--prioridad: invalid choice: 'URGENTE' (choose from 'BAJA', 'MEDIA', 'ALTA')


## 7. Programa principal y `logging`

Añadir `main(argv=None)`, que configura `logging` según el número de `-v` (`WARNING`, `INFO` o `DEBUG`, siempre a la salida de error para no mezclarse con el listado), carga las tareas, ejecuta la orden con `match` y las guarda si han cambiado. Devuelve el código de salida: 0 si todo va bien, 1 si el `id` no existe y 2 (lo hace `parser.error`) si la fecha no es válida. Terminar con el `if __name__ == "__main__"` y mostrar la ayuda ejecutando el programa con `subprocess` y `sys.executable`, el mismo Python del notebook.

In [14]:
%%writefile -a tareas.py


def main(argv=None):
    parser = crea_parser()
    args = parser.parse_args(argv)
    nivel = [logging.WARNING, logging.INFO, logging.DEBUG][min(args.verbose, 2)]
    logging.basicConfig(level=nivel, format="%(levelname)s: %(message)s", stream=sys.stderr, force=True)

    tareas = carga(args.fichero)
    try:
        match args.orden:
            case "añadir":
                try:
                    limite = lee_fecha(args.limite, args.hoy) if args.limite else None
                except ValueError:
                    parser.error(f"fecha no válida: {args.limite!r}")
                tarea = agrega(tareas, args.titulo, Prioridad[args.prioridad], limite)
                print(f"Añadida la tarea {tarea.id}.")
            case "listar":
                seleccion = [t for t in ordena(tareas)
                             if not (args.pendientes and t.hecha)
                             and args.prioridad in (None, t.prioridad.name)]
                for tarea in seleccion:
                    print(formatea(tarea, args.hoy))
                if not seleccion:
                    print("No hay tareas.")
                return 0
            case "completar":
                completa(tareas, args.id)
                print(f"Completada la tarea {args.id}.")
            case "borrar":
                borra(tareas, args.id)
                print(f"Borrada la tarea {args.id}.")
    except TareaNoEncontrada as error:
        log.error("%s", error)
        return 1
    guarda(tareas, args.fichero)
    return 0


if __name__ == "__main__":
    sys.exit(main())

Appending to tareas.py


In [15]:
def ejecuta(*argumentos):
    # -X utf8: escribe en UTF-8 aunque Windows use otra codificación; -u: sin búfer, para no desordenar
    # la salida normal y la de error al juntarlas
    orden = [sys.executable, "-X", "utf8", "-u", "tareas.py", "--hoy", HOY.isoformat(), *argumentos]
    resultado = subprocess.run(orden, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, encoding="utf-8")
    print("$ python tareas.py", " ".join(f'"{a}"' if " " in a else a for a in argumentos))
    print(resultado.stdout, end="")
    if resultado.returncode:
        print(f"(código de salida {resultado.returncode})")
    return resultado

ejecuta("-h")
ejecuta("añadir", "-h");

$ python tareas.py -h
usage: tareas [-h] [--fichero FICHERO] [--hoy HOY] [-v]
              {añadir,listar,completar,borrar} ...

Gestor de tareas de terminal.

positional arguments:
  {añadir,listar,completar,borrar}
    añadir              añade una tarea
    listar              muestra las tareas
    completar           marca una tarea como hecha
    borrar              borra una tarea

options:
  -h, --help            show this help message and exit
  --fichero FICHERO     fichero JSON con las tareas
  --hoy HOY             fecha de referencia
  -v, --verbose         más mensajes (-v, -vv)


$ python tareas.py

añadir -h
usage: tareas añadir [-h] [-p {BAJA,MEDIA,ALTA}] [-l LIMITE] titulo

positional arguments:
  titulo

options:
  -h, --help            show this help message and exit
  -p, --prioridad {BAJA,MEDIA,ALTA}
  -l, --limite LIMITE   AAAA-MM-DD, hoy, mañana o +N días


## 8. Una sesión completa

Ejecutar el programa varias veces seguidas con `subprocess`, como se haría en la terminal: añadir cuatro tareas con distintas prioridades y fechas límite, listarlas, completar una, borrar otra con `-v` para ver los mensajes de `logging`, listar solo las pendientes y provocar los dos errores (un `id` que no existe y una fecha no válida). Mostrar al final el JSON que ha quedado.

In [16]:
if os.path.exists("tareas.json"):
    os.remove("tareas.json")  # empezar con la lista vacía

ejecuta("añadir", "Repasar grafos", "-p", "alta", "-l", "+2")
ejecuta("añadir", "Comprar café", "-p", "baja", "-l", "hoy")
ejecuta("añadir", "Leer sobre dataclasses")
ejecuta("añadir", "Entregar la práctica", "--prioridad", "ALTA", "--limite", "2026-11-01")
ejecuta("listar");

$ python tareas.py

 añadir "Repasar grafos" -p alta -l +2
Añadida la tarea 1.


$ python tareas.py añadir "Comprar café" -p baja -l hoy
Añadida la tarea 2.
$ python tareas.py añadir "Leer sobre dataclasses"
Añadida la tarea 3.


$ python tareas.py

 añadir "Entregar la práctica" --prioridad ALTA --limite 2026-11-01
Añadida la tarea 4.
$ python tareas.py listar
[ ]  4  ALTA   Entregar la práctica  (vencida hace 5 días)
[ ]  2  BAJA   Comprar café  (vence hoy)
[ ]  1  ALTA   Repasar grafos  (vence en 2 días)
[ ]  3  MEDIA  Leer sobre dataclasses  (sin fecha)


In [17]:
ejecuta("completar", "2")
ejecuta("-v", "borrar", "3")
ejecuta("listar", "--pendientes")
ejecuta("listar", "-p", "baja")
ejecuta("completar", "9")
ejecuta("añadir", "Algo", "-l", "el lunes");

$ python tareas.py

 completar 2
Completada la tarea 2.


$ python tareas.py -v borrar 3
INFO: borrada la tarea 3
Borrada la tarea 3.


$ python tareas.py listar --pendientes
[ ]  4  ALTA   Entregar la práctica  (vencida hace 5 días)
[ ]  1  ALTA   Repasar grafos  (vence en 2 días)
$ python tareas.py listar -p baja
[x]  2  BAJA   Comprar café


$ python tareas.py

 completar 9
ERROR: no existe la tarea 9
(código de salida 1)
$ python tareas.py añadir Algo -l "el lunes"
usage: tareas [-h] [--fichero FICHERO] [--hoy HOY] [-v]
              {añadir,listar,completar,borrar} ...
tareas: error: fecha no válida: 'el lunes'
(código de salida 2)


In [18]:
with open("tareas.json", encoding="utf-8") as f:
    print(f.read())

[
  {
    "id": 1,
    "titulo": "Repasar grafos",
    "prioridad": "ALTA",
    "limite": "2026-11-08",
    "hecha": false
  },
  {
    "id": 2,
    "titulo": "Comprar café",
    "prioridad": "BAJA",
    "limite": "2026-11-06",
    "hecha": true
  },
  {
    "id": 4,
    "titulo": "Entregar la práctica",
    "prioridad": "ALTA",
    "limite": "2026-11-01",
    "hecha": false
  }
]



## 9. Tests con `pytest`

Escribir `tests_tareas.py` con tests de las fechas (con `parametrize`), de la ida y vuelta a JSON en un directorio temporal (`tmp_path`), de las operaciones y de la orden completa llamando a `main` con una lista de argumentos y capturando la salida con `capsys`, incluidos los dos códigos de error. Ejecutarlos con `pytest.main` y, al terminar, borrar todos los ficheros generados.

In [19]:
%%writefile tests_tareas.py
import json
from datetime import date

import pytest

import tareas
from tareas import Prioridad, Tarea

HOY = date(2026, 11, 6)


@pytest.mark.parametrize("texto, esperada", [
    ("hoy", HOY),
    ("Mañana", date(2026, 11, 7)),
    ("+30", date(2026, 12, 6)),
    ("2027-01-06", date(2027, 1, 6)),
])
def test_lee_fecha(texto, esperada):
    assert tareas.lee_fecha(texto, HOY) == esperada


@pytest.mark.parametrize("texto", ["ayer", "+", "-3", "2026-13-01"])
def test_lee_fecha_invalida(texto):
    with pytest.raises(ValueError):
        tareas.lee_fecha(texto, HOY)


def test_describe_limite():
    assert tareas.describe_limite(HOY, HOY) == "vence hoy"
    assert tareas.describe_limite(date(2026, 11, 7), HOY) == "vence en 1 día"
    assert tareas.describe_limite(date(2026, 11, 1), HOY) == "vencida hace 5 días"


def test_json_ida_y_vuelta(tmp_path):
    ruta = tmp_path / "t.json"
    lista = [Tarea(1, "Repasar", Prioridad.ALTA, HOY), Tarea(2, "Café ☕", hecha=True)]
    tareas.guarda(lista, ruta)
    assert tareas.carga(ruta) == lista
    assert json.loads(ruta.read_text(encoding="utf-8"))[0]["prioridad"] == "ALTA"
    assert tareas.carga(tmp_path / "no-existe.json") == []


def test_operaciones():
    lista = []
    for titulo in "abc":
        tareas.agrega(lista, titulo)
    tareas.borra(lista, 3)
    assert tareas.agrega(lista, "d").id == 3
    assert tareas.completa(lista, 1).hecha
    with pytest.raises(tareas.TareaNoEncontrada):
        tareas.borra(lista, 7)


def test_ordena():
    lista = [Tarea(1, "hecha", hecha=True), Tarea(2, "sin fecha", Prioridad.ALTA),
             Tarea(3, "baja", Prioridad.BAJA, HOY), Tarea(4, "alta", Prioridad.ALTA, HOY)]
    assert [t.id for t in tareas.ordena(lista)] == [4, 3, 2, 1]


def orden(tmp_path, *argumentos):
    return tareas.main(["--fichero", str(tmp_path / "t.json"), "--hoy", HOY.isoformat(), *argumentos])


def test_main(tmp_path, capsys):
    assert orden(tmp_path, "añadir", "Estudiar", "-p", "alta", "-l", "mañana") == 0
    assert orden(tmp_path, "añadir", "Pasear") == 0
    assert orden(tmp_path, "completar", "2") == 0
    capsys.readouterr()
    assert orden(tmp_path, "listar") == 0
    assert capsys.readouterr().out.splitlines() == [
        "[ ]  1  ALTA   Estudiar  (vence en 1 día)",
        "[x]  2  MEDIA  Pasear",
    ]


def test_main_errores(tmp_path, capsys):
    assert orden(tmp_path, "borrar", "7") == 1
    assert "no existe la tarea 7" in capsys.readouterr().err
    with pytest.raises(SystemExit) as error:
        orden(tmp_path, "añadir", "Algo", "-l", "ayer")
    assert error.value.code == 2
    assert "fecha no válida" in capsys.readouterr().err

Writing tests_tareas.py


In [20]:
import pytest

sys.modules.pop("tareas", None)
codigo = pytest.main(["-q", "-p", "no:cacheprovider", "--color=no", "tests_tareas.py"])
assert codigo == pytest.ExitCode.OK

.

.

.

.

.

.

... (6 outputs left out)

.

.

                                                           [100%]


14 passed in 0.05s


Borrar los ficheros generados.

In [21]:
for nombre in ["tareas.py", "tests_tareas.py", "tareas.json"]:
    if os.path.exists(nombre):
        os.remove(nombre)
shutil.rmtree("__pycache__", ignore_errors=True)
print(sorted(os.listdir()))

['01-problemas-de-entrevista.ipynb', '02-puzles.ipynb', '03-proyecto-gestor-de-tareas.ipynb']
